# 🎬 ניתוח גרף שיתופי הפעולה של הקולנוע הישראלי
## סיכום מקיף לסרטון הצגה (5 דקות)

**תקציר המחקר:** ניתוח מקיף של רשת שיתופי הפעולה בין 1,399 שחקנים ישראלים על פני 107 שנים (1918-2025), כולל זיהוי קהילות, ניתוח מרכזיות, ומודל למידת מכונה לניבוי שיתופי פעולה עתידיים.

---

## 📚 Part 1: ייבוא ספריות וטעינת נתונים

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from bidi.algorithm import get_display

sys.path.append('../src')
from graph_build import load_cast_edges, build_full_graph, _actor_id

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 12

def he(s):
    """Fix Hebrew text for matplotlib"""
    return "\n".join(get_display(line) for line in str(s).split("\n"))

print("✅ כל הספריות נטענו בהצלחה!")

In [ ]:
# Load data
cast_df = load_cast_edges('../data/processed/cast_edges.csv')
cast_df['actor_id'] = [
    _actor_id(slug, name) 
    for slug, name in zip(cast_df['actor_slug'], cast_df['actor_name'])
]

# Build full graph
G = build_full_graph(cast_df)

print(f"✅ הגרף נטען: {G.number_of_nodes():,} שחקנים, {G.number_of_edges():,} קשרים")

---
## 📊 Part 2: סטטיסטיקות מרכזיות - "המספרים המדברים"

### הנתונים המרשימים ביותר מהגרף

In [ ]:
# Calculate key statistics
n_actors = G.number_of_nodes()
n_edges = G.number_of_edges()
density = nx.density(G)
n_movies = cast_df['movie_slug'].nunique()
avg_degree = sum(dict(G.degree()).values()) / n_actors

# Get giant component
giant = max(nx.connected_components(G), key=len)
giant_size = len(giant)
giant_pct = 100 * giant_size / n_actors

# Create impressive statistics display
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(he('סטטיסטיקות מרכזיות - הגרף במספרים'), fontsize=20, fontweight='bold', y=0.98)

stats = [
    (n_actors, he('שחקנים'), '#3b6ea5'),
    (n_edges, he('שיתופי פעולה'), '#e8743b'),
    (n_movies, he('סרטים'), '#19a974'),
    (f"{avg_degree:.1f}", he('ממוצע שותפים'), '#a463f2'),
    (f"{giant_pct:.0f}%", he('ברכיב הענק'), '#f7b731'),
    (f"{density*10000:.2f}", he('צפיפות (×10⁻⁴)'), '#eb3b5a')
]

for ax, (value, label, color) in zip(axes.flat, stats):
    ax.text(0.5, 0.6, f"{value:,}" if isinstance(value, int) else value, 
            ha='center', va='center', fontsize=48, fontweight='bold', color=color)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=18, color='#333')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, 
                               fill=False, edgecolor=color, linewidth=3, alpha=0.7))

plt.tight_layout()
plt.savefig('../figures/presentation_key_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ סטטיסטיקות מרכזיות הוצגו!")

---
## 🌐 Part 3: ויזואליזציה מרשימה של הגרף

### רשת שיתופי הפעולה עם קהילות (Louvain)

In [ ]:
from communities import run_algorithms

# Run community detection
print("מזהה קהילות...")
results = run_algorithms(G, seed=42)
louvain_communities = results['louvain']

# Create node-to-community mapping
node_to_comm = {}
for idx, comm in enumerate(louvain_communities):
    for node in comm:
        node_to_comm[node] = idx

print(f"✅ זוהו {len(louvain_communities)} קהילות")

In [ ]:
# Create beautiful graph visualization
print("יוצר ויזואליזציה מרשימה של הגרף...")

# Get giant component for visualization
giant_nodes = max(nx.connected_components(G), key=len)
G_giant = G.subgraph(giant_nodes).copy()

# Sample for visualization (top degree nodes)
degrees = dict(G_giant.degree())
top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:300]
G_vis = G_giant.subgraph(top_nodes).copy()

# Create layout
print("מחשב מיקום צמתים...")
pos = nx.spring_layout(G_vis, seed=42, k=0.3, iterations=50)

# Assign colors by community
colors = [node_to_comm.get(node, 0) for node in G_vis.nodes()]
node_sizes = [degrees[node] * 30 for node in G_vis.nodes()]

# Plot
fig, ax = plt.subplots(figsize=(20, 20))
nx.draw_networkx_edges(G_vis, pos, ax=ax, alpha=0.15, width=0.5, edge_color='gray')
nx.draw_networkx_nodes(G_vis, pos, ax=ax, 
                       node_size=node_sizes, 
                       node_color=colors,
                       cmap=plt.cm.tab20,
                       alpha=0.85,
                       linewidths=1,
                       edgecolors='white')

# Add labels for top nodes
top_20 = sorted(degrees, key=degrees.get, reverse=True)[:20]
labels = {n: he(G.nodes[n].get('display_name', n).replace('_', ' ')) 
          for n in top_20 if n in G_vis.nodes()}
nx.draw_networkx_labels(G_vis, pos, labels, ax=ax, font_size=10, font_weight='bold')

ax.set_title(he(f'רשת שיתופי הפעולה - {len(louvain_communities)} קהילות'), 
             fontsize=24, fontweight='bold', pad=20)
ax.text(0.5, -0.05, he(f'מוצגים 300 השחקנים המרכזיים מתוך {n_actors:,} שחקנים'), 
        ha='center', fontsize=14, transform=ax.transAxes)
ax.axis('off')

plt.tight_layout()
plt.savefig('../figures/presentation_network_communities.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✅ ויזואליזציית הגרף הושלמה!")

---
## ⭐ Part 4: השחקנים המרכזיים ביותר

### Top 15 לפי Degree Centrality

In [ ]:
# Calculate centralities
degree_cent = nx.degree_centrality(G)
top_15 = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:15]

# Prepare data
names = [he(G.nodes[actor].get('display_name', actor).replace('_', ' ')) for actor, _ in top_15]
values = [G.degree(actor) for actor, _ in top_15]

# Create horizontal bar chart
fig, ax = plt.subplots(figsize=(14, 10))
bars = ax.barh(range(len(names)), values, color=plt.cm.viridis(np.linspace(0.3, 0.9, len(names))))

ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=13)
ax.invert_yaxis()
ax.set_xlabel(he('מספר שיתופי פעולה'), fontsize=14, fontweight='bold')
ax.set_title(he('15 השחקנים המרכזיים ביותר בקולנוע הישראלי'), 
             fontsize=18, fontweight='bold', pad=20)

# Add value labels
for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(val + 1, i, f'{val}', va='center', fontsize=11, fontweight='bold')

ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/presentation_top_actors.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ השחקנים המרכזיים הוצגו!")

---
## 📈 Part 5: התפתחות הגרף לאורך זמן

### השוואה בין 3 תקופות

In [ ]:
from graph_build import build_graph_for_years

# Build graphs for three periods
periods = [
    (1948, 1977, 'A', he('תקופה א: 1948-1977\n(הקולנוע המוקדם)')),
    (1978, 1989, 'B', he('תקופה ב: 1978-1989\n(תקופת המעבר)')),
    (1990, 2025, 'C', he('תקופה ג: 1990-2025\n(הקולנוע המודרני)'))
]

period_stats = []
for y_min, y_max, label, name in periods:
    G_period = build_graph_for_years(cast_df, y_min, y_max)
    period_stats.append({
        'name': name,
        'actors': G_period.number_of_nodes(),
        'collaborations': G_period.number_of_edges(),
        'density': nx.density(G_period) * 1000,
        'avg_degree': sum(dict(G_period.degree()).values()) / max(G_period.number_of_nodes(), 1)
    })

# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(he('התפתחות הגרף לאורך תקופות'), fontsize=20, fontweight='bold')

metrics = ['actors', 'collaborations', 'avg_degree', 'density']
titles = [he('מספר שחקנים'), he('מספר שיתופי פעולה'), he('ממוצע שותפים'), he('צפיפות (×10⁻³)')]
colors = ['#3b6ea5', '#e8743b', '#19a974']

for ax, metric, title in zip(axes.flat, metrics, titles):
    values = [s[metric] for s in period_stats]
    bars = ax.bar(range(3), values, color=colors, alpha=0.8, edgecolor='white', linewidth=2)
    
    ax.set_xticks(range(3))
    ax.set_xticklabels([s['name'] for s in period_stats], fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:,.0f}' if val >= 10 else f'{val:.1f}',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/presentation_temporal_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ התפתחות לאורך זמן הוצגה!")

---
## 🎯 Part 6: מודל הניבוי - הצלחות וכשלונות

### תוצאות המודל המלא

In [ ]:
# Load prediction results
results_df = pd.read_csv('../data/processed/future_predictions_2021_2025_single_horizon.csv')

# Calculate key metrics
tp = len(results_df[results_df['actually_connected'] == 1])
fp = len(results_df[results_df['actually_connected'] == 0])
precision = tp / (tp + fp) if (tp + fp) > 0 else 0

# Create results visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(he('תוצאות מודל הניבוי - הצלחות מרשימות'), fontsize=18, fontweight='bold')

# Chart 1: Precision
ax = axes[0]
ax.bar([0], [precision*100], color='#19a974', alpha=0.8, width=0.5)
ax.set_ylim(0, 100)
ax.set_xlim(-0.5, 0.5)
ax.set_xticks([0])
ax.set_xticklabels([he('Precision')])
ax.set_ylabel(he('אחוז (%)'), fontsize=12)
ax.set_title(he(f'דיוק הניבויים\n{precision*100:.1f}%'), fontsize=14, fontweight='bold')
ax.text(0, precision*100 + 2, f'{precision*100:.1f}%', ha='center', fontsize=20, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Chart 2: Hits breakdown
ax = axes[1]
categories = [he('ניבויים נכונים'), he('ניבויים שגויים')]
values = [tp, fp]
colors_pie = ['#19a974', '#e8743b']
wedges, texts, autotexts = ax.pie(values, labels=categories, colors=colors_pie, autopct='%1.0f%%',
                                    startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax.set_title(he(f'התפלגות התוצאות\n(מתוך {tp+fp} ניבויים)'), fontsize=14, fontweight='bold')

# Chart 3: Score distribution
ax = axes[2]
connected = results_df[results_df['actually_connected'] == 1]['score']
not_connected = results_df[results_df['actually_connected'] == 0]['score']
ax.hist([connected, not_connected], bins=20, label=[he('התממשו'), he('לא התממשו')],
        color=['#19a974', '#e8743b'], alpha=0.7, edgecolor='white')
ax.set_xlabel(he('ציון חיזוי'), fontsize=12)
ax.set_ylabel(he('תדירות'), fontsize=12)
ax.set_title(he('התפלגות ציוני החיזוי'), fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/presentation_model_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ תוצאות המודל: {tp}/{tp+fp} ניבויים נכונים ({precision*100:.1f}% precision)")

In [ ]:
# Show successful predictions (hits)
hits = results_df[results_df['actually_connected'] == 1].sort_values('score', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(14, 8))
names = [he(f"{row['u'].replace('_', ' ')} ו{row['v'].replace('_', ' ')}") 
         for _, row in hits.iterrows()]
scores = hits['score'].values

bars = ax.barh(range(len(names)), scores, color='#19a974', alpha=0.8, edgecolor='white', linewidth=2)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=12)
ax.invert_yaxis()
ax.set_xlabel(he('ציון חיזוי'), fontsize=13)
ax.set_title(he('10 הניבויים המוצלחים ביותר - שיתופי פעולה שהתממשו!'), 
             fontsize=16, fontweight='bold', pad=15)

for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(score + 0.01, i, f'{score:.3f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.05)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/presentation_successful_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ הצלחות המודל הוצגו!")

In [ ]:
# Show failed predictions (interesting misses)
misses = results_df[results_df['actually_connected'] == 0].sort_values('score', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(14, 8))
names = [he(f"{row['u'].replace('_', ' ')} ו{row['v'].replace('_', ' ')}") 
         for _, row in misses.iterrows()]
scores = misses['score'].values

bars = ax.barh(range(len(names)), scores, color='#e8743b', alpha=0.8, edgecolor='white', linewidth=2)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=12)
ax.invert_yaxis()
ax.set_xlabel(he('ציון חיזוי'), fontsize=13)
ax.set_title(he('כשלונות מעניינים - ציפינו לשיתופי פעולה שלא התממשו'), 
             fontsize=16, fontweight='bold', pad=15)

for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(score + 0.01, i, f'{score:.3f}', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 1.05)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/presentation_failed_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ כשלונות המודל הוצגו (זוגות עם פוטנציאל גבוה שטרם שיתפו פעולה)!")

---
## 💡 Part 7: תובנות מרכזיות

### מה למדנו על תעשיית הקולנוע הישראלית?

In [ ]:
# Create insights summary
fig = plt.figure(figsize=(16, 10))
fig.suptitle(he('תובנות מרכזיות מהמחקר'), fontsize=22, fontweight='bold', y=0.96)

insights = [
    ("🌐", he("רשת מקושרת"), 
     he(f"{giant_pct:.0f}% מהשחקנים ברכיב ענק אחד\nמעיד על תעשייה מקושרת היטב")),
    
    ("📈", he("צמיחה מרשימה"), 
     he(f"תקופה ג׳ (1990-2025):\n{period_stats[2]['actors']:,} שחקנים, פי-{period_stats[2]['actors']/period_stats[0]['actors']:.1f} מתקופה א׳")),
    
    ("👥", he("קהילות מובחנות"), 
     he(f"{len(louvain_communities)} קהילות זוהו\nמעידות על סגנונות ז׳אנרים שונים")),
    
    ("⭐", he("שחקנים מרכזיים"), 
     he(f"השחקן המרכזי ביותר:\n{top_15[0][0].replace('_', ' ')}\n({values[0]} שיתופי פעולה)")),
    
    ("🎯", he("מודל מוצלח"), 
     he(f"Precision של {precision*100:.1f}%\nבניבוי שיתופי פעולה עתידיים")),
    
    ("💎", he("פוטנציאל עתידי"), 
     he(f"זוהו {fp} זוגות עם פוטנציאל גבוה\nשטרם שיתפו פעולה"))
]

for idx, (emoji, title, text) in enumerate(insights):
    ax = fig.add_subplot(2, 3, idx+1)
    
    ax.text(0.5, 0.75, emoji, ha='center', va='center', fontsize=60)
    ax.text(0.5, 0.50, title, ha='center', va='center', 
            fontsize=16, fontweight='bold', color='#2c3e50')
    ax.text(0.5, 0.20, text, ha='center', va='center', 
            fontsize=12, color='#34495e', multialignment='center')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    colors_box = ['#3b6ea5', '#e8743b', '#19a974', '#a463f2', '#f7b731', '#eb3b5a']
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, 
                               fill=True, facecolor=colors_box[idx], 
                               alpha=0.1, edgecolor=colors_box[idx], linewidth=3))

plt.tight_layout()
plt.savefig('../figures/presentation_insights.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print("\n✅ תובנות מרכזיות הוצגו!")

---
## 🎬 סיכום - הכל מוכן לסרטון!

### קבצי תמונות שנוצרו:
1. `presentation_key_stats.png` - סטטיסטיקות מרכזיות
2. `presentation_network_communities.png` - ויזואליזציית הגרף עם קהילות
3. `presentation_top_actors.png` - 15 השחקנים המרכזיים
4. `presentation_temporal_evolution.png` - התפתחות לאורך זמן
5. `presentation_model_results.png` - תוצאות המודל
6. `presentation_successful_predictions.png` - הצלחות המודל
7. `presentation_failed_predictions.png` - כשלונות מעניינים
8. `presentation_insights.png` - תובנות מרכזיות

### המלצה לסדר הצגה בסרטון (5 דקות):
1. **0:00-0:30** - הצגת הנושא + סטטיסטיקות מרכזיות
2. **0:30-1:30** - ויזואליזציה מרשימה של הגרף + הסבר על קהילות
3. **1:30-2:15** - שחקנים מרכזיים + התפתחות לאורך זמן
4. **2:15-3:30** - תוצאות מודל הניבוי (הצלחות וכשלונות)
5. **3:30-4:45** - תובנות מרכזיות מהמחקר
6. **4:45-5:00** - סיכום וסגירה

---
**כל הקבצים נשמרו בתיקייה:** `israeli_actors_graph/figures/`

**בהצלחה בהצגה! 🎉**